# Filter Electrical Conductivity Papers

This notebook filters the pre-existing `thermocatalysis_keywords_and_LLM` split (1333 papers) with new keywords for electrical conductivity.

In [2]:
from datasets import load_dataset

# Load the pre-filtered split (1333 papers)
dataset = load_dataset(
    "LeMaterial/LeMat-Synth-Papers",
    "full",
    split="thermocatalysis_keywords_and_LLM",
    token=True,
)

print(f"Loaded {len(dataset)} papers")
print(f"Columns: {dataset.column_names}")

Loaded 1333 papers
Columns: ['id', 'title', 'authors', 'abstract', 'doi', 'published_date', 'updated_date', 'categories', 'license', 'pdf_url', 'views_count', 'read_count', 'citation_count', 'keywords', 'text_paper', 'text_si', 'source', 'pdf_extractor', 'images', 'structured_synthesis']


In [4]:
# Preview the data
dataset.to_pandas()[["id", "title", "pdf_url"]].head()

,id,title,pdf_url
0,0706.3760,Correlation between transport properties and l...,http://arxiv.org/pdf/0706.3760v1
1,0707.0522,Superconductivity in spinel oxide LiTi2O4 epit...,http://arxiv.org/pdf/0707.0522v2
2,0707.1630,Direct link between low temperature magnetism ...,http://arxiv.org/pdf/0707.1630v2
3,0707.2518,Positive and negative pressure effects on the ...,http://arxiv.org/pdf/0707.2518v1
4,0708.1135,Charge transport in arrays of PbSe nanocrystals,http://arxiv.org/pdf/0708.1135v2


In [5]:
def filter_dataset(ds, text_column, list_keywords, exclude_keywords=None):
    """Filter dataset by keywords in a text column."""

    def matches_any_keyword(example):
        if example[text_column] is None:
            return False
        text_lower = example[text_column].lower()
        return any(kw.lower() in text_lower for kw in list_keywords)

    filtered = ds.filter(matches_any_keyword)

    if exclude_keywords:

        def matches_exclude(example):
            if example[text_column] is None:
                return False
            text_lower = example[text_column].lower()
            return any(ek.lower() in text_lower for ek in exclude_keywords)

        filtered = filtered.filter(lambda x: not matches_exclude(x))

    return filtered

In [ ]:
# Define keywords for electrical conductivity
text_column = "abstract"

list_keywords = [
    # Core conductivity terms
    "electrical conductivity",
    "electronic conductivity",
    # "ionic conductivity",
    # "electrical resistivity",
    # "sheet resistance",
    # "conductance",
    # Measurement-related
    # "four-point probe",
    # "four point probe",
    # "van der Pauw",
    # "impedance spectroscopy",
    # "electrochemical impedance",
    # "EIS",
    # Units/values
    "S/cm",
    "S cm",
    "siemens",
    # Material properties
    # "semiconductor",
    # "semiconducting",
    # "charge carrier",
    # "carrier mobility",
    # "electron mobility",
    # "hole mobility",
    # "band gap",
    # "bandgap",
    # "doping",
    # "doped",
    # "thermoelectric",
    # "solid electrolyte",
    # "ion conductor",
    # "proton conductor",
    # "superconductor",
    # "superconducting",
]

exclude_keywords = [
    "thermal conductivity",
    "photocatalysis",
]

# Apply filter
filtered_dataset = filter_dataset(
    dataset, text_column, list_keywords, exclude_keywords
)

print(f"Filtered: {len(filtered_dataset)} / {len(dataset)} papers")

Filter:   0%|          | 0/1333 [00:00<?, ? examples/s]

Filter:   0%|          | 0/27 [00:00<?, ? examples/s]

Filtered: 27 / 1333 papers


In [11]:
# Preview filtered results
filtered_dataset.to_pandas()[["id", "title", "pdf_url"]].head(100)

,id,title,pdf_url
0,0812.2747,"High-temperature oxygen non-stoichiometry, con...",http://arxiv.org/pdf/0812.2747v1
1,0910.4508,Fast-ion conduction and flexibility and rigidi...,http://arxiv.org/pdf/0910.4508v1
2,1007.1757,Dielectric properties of Li2O-3B2O3 glasses,http://arxiv.org/pdf/1007.1757v1
3,1007.5410,Electrical Relaxation and Transport in 0.5Cs2O...,http://arxiv.org/pdf/1007.5410v1
4,1201.1153,Tunable electrical transport through annealed ...,http://arxiv.org/pdf/1201.1153v1
5,1202.2394,Mechanism of small-polaron formation in the bi...,http://arxiv.org/pdf/1202.2394v1
6,1312.1086,Physical characteristics and cation distributi...,http://arxiv.org/pdf/1312.1086v1
7,1502.05078,Paradox of Peroxy Defects and Positive Holes i...,http://arxiv.org/pdf/1502.05078v1
8,1701.05366,Comparative study of LaNiO$_3$/LaAlO$_3$ heter...,http://arxiv.org/pdf/1701.05366v1
9,1703.06196,Understanding the High Temperature Thermoelect...,http://arxiv.org/pdf/1703.06196v1


In [12]:
# Check which keywords matched (optional - for analysis)
def count_keyword_matches(ds, text_column, keywords):
    counts = {}
    for kw in keywords:

        def has_keyword(example, keyword=kw):
            if example[text_column] is None:
                return False
            return keyword.lower() in example[text_column].lower()

        count = len(ds.filter(has_keyword))
        if count > 0:
            counts[kw] = count
    return dict(sorted(counts.items(), key=lambda x: -x[1]))


keyword_counts = count_keyword_matches(dataset, text_column, list_keywords)
print("Keyword match counts (before exclusion):")
for kw, count in keyword_counts.items():
    print(f"  {kw}: {count}")

Keyword match counts (before exclusion):
  electrical conductivity: 35
  electronic conductivity: 3


In [ ]:
# Push to HuggingFace as a new split (creates a PR)
filtered_dataset.push_to_hub(
    repo_id="LeMaterial/LeMat-Synth-Papers",
    config_name="full",
    split="electrical_conductivity_keywords_only",  # New split name
    token=True,
    create_pr=True,
    commit_message="Add electrical_conductivity_keywords_only split filtered from thermocatalysis_keywords_and_LLM",
)

print("Created PR for new split 'electrical_conductivity_keywords_only'!")